# Import data

## Import building data
Read from *buildings.json* and import as `buildings_data` Dataframe with `building_id` as index.

In [1]:
import pandas as pd

# -------------------------- LOAD DATA --------------------------- #

BUILDING_FILENAME: str = "../data/buildings.json"
buildings_df: pd.DataFrame = pd.read_json(BUILDING_FILENAME).set_index("building_id")

print(buildings_df)

                   name door_orientation validation_color validation_symbol  \
building_id                                                                   
L1-01           Butcher            South              red            square   
L1-02              Barn             East           orange            circle   
L1-03            Potter             East            white          triangle   
L1-04          Windmill            North            green              star   
L1-05        Blacksmith             East            brown              moon   
L1-06            Tavern            South             pink             cross   
L1-07            Bakery             West           purple             arrow   
L1-08            Tailor            North           yellow        semicircle   
L1-09            Cooper             West             blue             cloud   

             grid_pos_x  grid_pos_z  world_pos_x  world_pos_z  
building_id                                                    
L1

## Import attempts data (savefile_04)

### Import data
Read from *savefile_04_events.json* and import as `game_data` Dataframe.
For now, saving all challenges data in a single item `challengesList` in the same `challengesSaveObject`. In the future might consider splitting per challenge.

⚠️ Have not tested game save with multiple challenges.

 ### Clean up data
1. if there is a "finished" line for a given challenge_id, attempt_number combination, delete the "started" row, aka keep the latest line
2. If the attempt duration value is 0, change it Nan. It means that the attempt was started but not finished.


In [5]:
import pandas as pd
import json
import datetime as dt
import numpy as np


GAME_FILENAME: str = "../data/savefile_04_events.json"


# -------------------------- LOAD DATA --------------------------- #

# 1. Load JSON
with open(GAME_FILENAME, mode="r") as file:
    game_data: json = json.load(file)

# 2. Extract top-level metadata
player_id: str = game_data["playerId"]
save_time: dt.datetime = dt.datetime.strptime(game_data["saveDateTime"], "%Y-%m-%d %H:%M:%S")
game_duration: float = float(game_data["challengesSaveObject"]["challengesList"][0]["gameTimer"])

# 3. Build rows from the list
challenge_data_rows: list[dict] = []

challenge_id: str = game_data["challengesSaveObject"]["challengesList"][0]["challengeId"]
challenge_duration: float = float(game_data["challengesSaveObject"]["challengesList"][0]["challengeDuration"])

for entry in game_data["challengesSaveObject"]["challengesList"][0]["attemptsList"]:
    challenge_data_rows.append({
        "start_time": float(entry["startTime"]),
        "challenge_id": challenge_id,
        "target_building_id": entry["targetBuildingId"],
        "attempt_number": entry["attemptNumber"],
        "attempt_state": entry["state"],
        "attempt_duration": entry["attemptDuration"],
        "challenge_duration": challenge_duration,
        "game_duration": game_duration,
        "player_id": player_id
    })

challenge_df: pd.DataFrame = pd.DataFrame(challenge_data_rows)

# -------------------------- CLEAN UP DATA --------------------------- #

challenge_df = challenge_df.drop_duplicates(subset=["challenge_id", "attempt_number"], keep="last")
challenge_df["attempt_duration"] = challenge_df["attempt_duration"].replace(0, np.nan)

print(challenge_df)

   start_time challenge_id target_building_id  attempt_number attempt_state  \
1    6.059720            1              L1-01               1      finished   
3   67.631180            1              L1-01               2      finished   
5  121.890045            1              L1-01               3      finished   
6  184.524246            1              L1-01               4       started   

   attempt_duration  challenge_duration  game_duration player_id  
1         58.800755          183.319214     193.958817    id_001  
3         52.341530          183.319214     193.958817    id_001  
5         61.067085          183.319214     193.958817    id_001  
6               NaN          183.319214     193.958817    id_001  


## Functions

In [3]:
# -------------------------- FUNCTIONS --------------------------- #

def get_lowest_time_value_index(time: float) -> int:
   """
   Returns the lowest daframe index for a game time.
   If there is no recorded data for this play time, returns '-1'
    """
   try:
      lowest_time_value_index: int = challenge_df[(challenge_df["start_time"] < time)]["start_time"].idxmax()
   except ValueError:
      return -1
   else:
      return lowest_time_value_index


def get_attempt_num_for_time(time: float) -> int:
   """
   Returns the attempt number for a given play time.
   If there is no attempt number for the play time, returns '-1'
   """
   lowest_time_value_index: int = get_lowest_time_value_index(time)

   if lowest_time_value_index == -1:
      return -1
   else:
      return int(challenge_df.loc[lowest_time_value_index]["attempt_number"]) #  without casting, returns np.int64(2)


def get_target_building_id_for_time(time: float) -> str:
   """
   Returns the attempt number for a given play time.
   If there is no attempt number for the play time, returns '-1'
   """
   lowest_time_value_index: int = get_lowest_time_value_index(time)

   if lowest_time_value_index == -1:
      return -1
   else:
      return challenge_df.loc[lowest_time_value_index]["target_building_id"] #  casting not required for strings

# TODO: exception for from value
def get_value_for_key_and_time(key: str, time: float) -> str:
   """
   Returns the attempt number for a given play time.
   If there is no attempt number for the play time, returns '-1'
   """
   lowest_time_value_index: int = get_lowest_time_value_index(time)

   if lowest_time_value_index == -1:
      return -1
   else:
      return challenge_df.loc[lowest_time_value_index][key] #  casting not required for strings


#print(get_attempt_num_for_time(3))
#print(get_target_building_id_for_time(68))

## Import Player Movement

In [10]:
# 3. Build rows from the list, reshaping position/rotation into tuples
player_movement_rows: list[dict] = []

for entry in game_data["playerSaveObject"]["playerMovementSaveObject"]["playerMovementInfoList"]:
    time: float = float(entry["gameTime"])
    target_building_id: str = get_target_building_id_for_time(time)
    attempt_numnber: int = get_attempt_num_for_time(time)

    player_movement_rows.append({
        "time": time,
        "challenge_id": get_value_for_key_and_time(key="challenge_id", time=time),
        "target_building_id": target_building_id,
        "target_building_name": buildings_df.loc[target_building_id, "name"],
        "attempt_number": attempt_numnber,
        "attempt_state": challenge_df.loc[
                            (challenge_df["challenge_id"] == challenge_id) & (challenge_df["attempt_number"] == attempt_numnber),
                            "attempt_state"
                        ].values[0],
        "attempt_duration": challenge_df.loc[
                                (challenge_df["challenge_id"] == challenge_id) & (challenge_df["attempt_number"] == attempt_numnber),
                                "attempt_duration"
                            ].values[0],
        "playerId": player_id,
        "pos_x": entry["position"]["x"],
        "pos_z": entry["position"]["z"],
        "rot_y": entry["rotation"]["y"]
    })

# 4. Create the DataFrame
player_movement_df: pd.DataFrame = pd.DataFrame(player_movement_rows)

x_list: list[float] = player_movement_df["pos_x"]
z_list: list[float] =  player_movement_df["pos_z"]
rot_y_list: list[float] = player_movement_df["pos_z"]
t_list: list[float] = player_movement_df["pos_z"]

print(player_movement_df)

           time challenge_id target_building_id target_building_name  \
0      9.514594            1              L1-01              Butcher   
1     10.514701            1              L1-01              Butcher   
2     11.514721            1              L1-01              Butcher   
3     12.522999            1              L1-01              Butcher   
4     13.531087            1              L1-01              Butcher   
..          ...          ...                ...                  ...   
180  190.341354            1              L1-01              Butcher   
181  191.350266            1              L1-01              Butcher   
182  192.357956            1              L1-01              Butcher   
183  193.357971            1              L1-01              Butcher   
184  193.958817            1              L1-01              Butcher   

     attempt_number attempt_state  attempt_duration playerId      pos_x  \
0                 1      finished         58.800755   id_001

# Model with charts

## Map configuration for Level 1

In [11]:
# MAP SIZE
GRID_COLS: int = 6
GRID_ROWS: int = 5
BLOCK_SIZE: int = 30

# The extent is computed — maps pixel corners to world coordinates
map_extent = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 0) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_x_offset: int = BLOCK_SIZE / 2
map_z_offset: int = BLOCK_SIZE / 2

map_coord: list[int] = [
    -BLOCK_SIZE / 2,
    (GRID_COLS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
    -BLOCK_SIZE / 2,
    (GRID_ROWS - 1) * BLOCK_SIZE + BLOCK_SIZE / 2,
]

map_coord[0] += map_x_offset  # left
map_coord[1] += map_x_offset  # right
map_coord[2] += map_z_offset  # bottom
map_coord[3] += map_z_offset  # top

## Map Graph

### Map Graph (plotly)

In [13]:
import plotly.graph_objects as go
from PIL import Image
import numpy as np

# Load map image
map_img = Image.open("../images/map_level1.png")

SIZE = 4
base_triangle = np.array([
    [0, SIZE/2],
    [-SIZE/2, -SIZE/2],
    [SIZE/2, -SIZE/2],
])

# --- Build the figure with initial traces ---

fig = go.Figure()

# Trace 0: trail (static — never changes)
fig.add_trace(go.Scatter(
    x=x_list, y=z_list,
    mode='markers',
    marker=dict(color='white', size=5, opacity=0.25),
    name='Trail',
    showlegend=False
))

# Trace 1: buildings with labels (static — toggle via legend click)
fig.add_trace(go.Scatter(
    x=buildings_df["world_pos_x"].tolist(),
    y=buildings_df["world_pos_z"].tolist(),
    mode='markers+text',
    marker=dict(color='orange', size=12, symbol='square',
                line=dict(color='black', width=1)),
    text=buildings_df["name"].tolist(),
    textposition='top right',
    textfont=dict(size=9, color='black'),
    name='POIs'
))

# Trace 2: path (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='lines+markers',
    marker=dict(color='cyan', size=4),
    line=dict(color='cyan', width=1),
    showlegend=False
))

# Trace 3: current position marker (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='markers',
    marker=dict(color='red', size=12),
    showlegend=False
))

# Trace 4: rotation triangle (animated — starts empty)
fig.add_trace(go.Scatter(
    x=[], y=[],
    fill='toself',
    fillcolor='red',
    line=dict(color='white', width=1.5),
    showlegend=False
))

# --- Build all animation frames upfront ---

frames = []
for i in range(len(x_list)):
    # Rotation triangle
    next_frame = min(i + 1, len(x_list) - 1)
    angle = -rot_y_list[i]
    rad = np.radians(angle)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    rotated = np.array([
        [v[0]*cos_a - v[1]*sin_a, v[0]*sin_a + v[1]*cos_a]
        for v in base_triangle
    ])
    rotated[:, 0] += x_list[next_frame]
    rotated[:, 1] += z_list[next_frame]
    tri_x = list(rotated[:, 0]) + [rotated[0, 0]]  # close the shape
    tri_y = list(rotated[:, 1]) + [rotated[0, 1]]

    frames.append(go.Frame(
        data=[
            go.Scatter(x=x_list[:i+1], y=z_list[:i+1]),      # path
            go.Scatter(x=[x_list[i]], y=[z_list[i]]),          # current
            go.Scatter(x=tri_x, y=tri_y),                     # triangle
        ],
        traces=[2, 3, 4],  # which trace indices to update
        name=str(i)
    ))

fig.frames = frames

# --- Background image ---

fig.add_layout_image(
    source=map_img,
    xref="x", yref="y",
    x=map_coord[0],
    y=map_coord[3],       # plotly anchors images from top-left
    sizex=map_coord[1] - map_coord[0],
    sizey=map_coord[3] - map_coord[2],
    sizing="stretch",
    layer="below"
)

# --- Layout: axes, play button, slider ---

fig.update_layout(
    width=800, height=700,
    title="Player Position over Time",
    xaxis=dict(range=[map_extent[0], map_extent[1]], dtick=30, showgrid=True),
    yaxis=dict(range=[map_extent[2], map_extent[3]], dtick=30,
               showgrid=True, scaleanchor="x"),

    # Play/pause buttons
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        x=0.5, y=-0.05, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=100, redraw=True),
                                  fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )],

    # Time slider
    sliders=[dict(
        active=0,
        x=0.05, len=0.9,
        currentvalue=dict(prefix="Time: "),
        steps=[
            dict(
                args=[[str(i)], dict(frame=dict(duration=0, redraw=True),
                                      mode="immediate")],
                method="animate",
                label=f"{t_list[i]:.1f}s"
            )
            for i in range(len(x_list))
        ]
    )]
)

fig.show()

In [ ]:
import plotly.express as px

# Filter out -1
df_filtered = player_movement_df[player_movement_df["attempt_number"] != -1]

fig2 = px.line(
    df_filtered.sort_values("time"),        # ensure chronological order
    x="pos_x",
    y="pos_z",
    color="attempt_number",
    #hover_data=["time", "target_building_id"], # can add columns this, remove is manual
    hover_data={
      "pos_x": False,
      "pos_z": False,
    },
    labels={
      "attempt_number": "attempt number" 
    },
    markers=True,                   # dots + lines
    title="Player Path by Attempt"
)

# Lock axes so toggling attempts doesn't rescale
fig2.update_layout(
   xaxis=dict(range=[map_extent[0], map_extent[1]], dtick=30, showgrid=True),
   yaxis=dict(range=[map_extent[2], map_extent[3]], dtick=30,
              showgrid=True, scaleanchor="x"),
   width=800,
   height=700,
   updatemenus=[dict(
     type="buttons",
      showactive=True,
      x=1.005, y=.75, xanchor="left",
      buttons=[
         dict(
            label="Toggle Map",
            method="relayout",
            args=[{"images[0].visible": True}],
            args2=[{"images[0].visible": False}],  # second click
         )
      ]
   )]
)

# Background image (ready for overlay)
fig2.add_layout_image(
    source=map_img,
    xref="x", yref="y",
    x=map_coord[0],
    y=map_coord[3],
    sizex=map_coord[1] - map_coord[0],
    sizey=map_coord[3] - map_coord[2],
    sizing="stretch",
    layer="below"
)

fig2.add_annotation(
    text= "Challenge "+ challenge_id,
    x=1.15, y=1.15,
    xref="paper", yref="paper",
    showarrow=False,
    font=dict(size=14),
    name="challenge_label"
)

fig2.show()